<a href="https://colab.research.google.com/github/Thinujan-Thillaiselvan/Statistical-Learning-e21411/blob/main/Assignment_5_Introduction_to_Numerical_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundations of Statistical Inference & Hypothesis Testing – Worked Solutions

This notebook walks through the theoretical parts in a friendly, step‑by‑step way and then implements the numerical MANOVA verification in code.

## Part A – Maximum Likelihood, Bias and Decision Space

### A.1 Bias of the MLE covariance and Bessel correction

We observe i.i.d. random vectors $\mathbf{X}_1, \dots, \mathbf{X}_n \in \mathbb{R}^m$ with mean $\boldsymbol{\mu}$ and covariance $\boldsymbol{\Sigma}$. Define the population‑centred variables

$$\mathbf{Y}_i = \mathbf{X}_i - \boldsymbol{\mu}.$$

The (raw) maximum likelihood estimator (MLE) of the covariance matrix under a multivariate Gaussian model is

$$\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T,$$
where the sample mean is

$$\widehat{\boldsymbol{\mu}}_n = \frac{1}{n} \sum_{i=1}^n \mathbf{X}_i.$$

First, rewrite $\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n$ in terms of the centred variables $\mathbf{Y}_i$.

Because

$$\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu} = \frac{1}{n} \sum_{j=1}^n (\mathbf{X}_j - \boldsymbol{\mu}) = \frac{1}{n} \sum_{j=1}^n \mathbf{Y}_j = \bar{\mathbf{Y}},$$

we have

$$\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n = (\mathbf{X}_i - \boldsymbol{\mu}) - (\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) = \mathbf{Y}_i - \bar{\mathbf{Y}},$$

where $\bar{\mathbf{Y}} = \tfrac{1}{n} \sum_{j=1}^n \mathbf{Y}_j$.

Therefore we can write the MLE purely in terms of centred variables:

$$\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n} \sum_{i=1}^n (\mathbf{Y}_i - \bar{\mathbf{Y}})(\mathbf{Y}_i - \bar{\mathbf{Y}})^T.$$

Now expand the sum inside the expectation. A standard result (which can be derived by expanding the square and using linearity of expectation, $\mathbb{E}[\mathbf{Y}_i]=\mathbf{0}$ and independence for $i\neq j$) is

$$\mathbb{E}\left[\sum_{i=1}^n (\mathbf{Y}_i - \bar{\mathbf{Y}})(\mathbf{Y}_i - \bar{\mathbf{Y}})^T\right] = (n-1)\boldsymbol{\Sigma}.$$

Dividing by $n$ gives the bias of the raw MLE:

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{n-1}{n} \boldsymbol{\Sigma}.$$

So the MLE systematically underestimates the true covariance by the factor $\tfrac{n-1}{n}$.

To remove this bias we keep exactly the same sum of squared deviations but change the scaling factor from $\tfrac{1}{n}$ to $\tfrac{1}{n-1}$:

$$\mathbf{S} = \frac{1}{n-1} \sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T.$$

Using the previous expectation,

$$\mathbb{E}[\mathbf{S}] = \frac{1}{n-1} \mathbb{E}\left[\sum_{i=1}^n (\mathbf{Y}_i - \bar{\mathbf{Y}})(\mathbf{Y}_i - \bar{\mathbf{Y}})^T\right] = \frac{1}{n-1} (n-1) \boldsymbol{\Sigma} = \boldsymbol{\Sigma}.$$

This factor $\tfrac{1}{n-1}$ is **Bessel's correction** in the multivariate setting: it compensates for the loss of one degree of freedom caused by estimating the mean from the same sample, and yields an unbiased estimator of the population covariance.

### A.2 Type I and Type II errors in continuous SHM

We consider a structural health monitoring (SHM) system where the null hypothesis $H_0$ represents a completely healthy structure, and the alternative $H_1$ represents some form of damage or abnormal behaviour. A diagnostic statistic (for example a Mahalanobis distance or Hotelling's $T^2$) is compared with a threshold derived from a reference distribution.

- **Type I error ($\alpha$)**: The system raises an alarm and declares damage **even though the structure is actually healthy**. In practice this is a false alarm or false shutdown: the plant is interrupted or inspected unnecessarily.
- **Type II error ($\beta$)**: The system **fails to raise an alarm when damage is truly present**. This is a missed detection event: the structure is deteriorating, but the monitoring layer does not notice it.

The **power** of the diagnostic test is $1-\beta$, the probability of correctly detecting damage when it occurs.

Now suppose the engineer chooses an ultra‑conservative significance level, for example $\alpha = 0.0001$, to make false alarms extremely rare. For a multivariate Gaussian baseline, the healthy region can be written as

$$\{\mathbf{x}: (\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) \le c_{1-\alpha}\},$$

where $c_{1-\alpha}$ is a high quantile of a $\chi^2_m$ or related distribution. When we decrease $\alpha$, the quantile $c_{1-\alpha}$ **increases**, so the ellipsoid grows and occupies a larger volume in feature space. More extreme points are still classified as "healthy".

The consequences are:

- The false alarm rate (Type I error) decreases by design, which protects against unnecessary shutdowns.
- At the same time, the power $1-\beta$ decreases: many moderately shifted but genuinely damaged states now lie inside the enlarged ellipsoid and are misclassified as healthy, increasing the Type II error probability.

So, making $\alpha$ extremely small stretches the healthy‑operation confidence ellipsoid, buying robustness against false alarms at the cost of sensitivity to real structural changes.

## Part B – Slutsky's Theorem and Asymptotic Mean Model

### B.1 Slutsky's Theorem (rigorous statement)

Let $(X_n)_{n\ge1}$ and $(Y_n)_{n\ge1}$ be sequences of random vectors in $\mathbb{R}^k$, and let $C$ be a constant vector (or matrix of compatible dimension).

Assume

- $X_n \xrightarrow{d} X$ (convergence in distribution),
- $Y_n \xrightarrow{p} C$ (convergence in probability).

Then Slutsky's Theorem states that:

1. $X_n + Y_n \xrightarrow{d} X + C$.
2. $X_n Y_n \xrightarrow{d} X C$ (matrix‑vector product in the multivariate case).
3. If each $Y_n$ is invertible with probability tending to 1 and $C$ is invertible, then   $Y_n^{-1} \xrightarrow{p} C^{-1}$ and $Y_n^{-1} X_n \xrightarrow{d} C^{-1} X$.

In words: when you study the limiting distribution of a statistic, you may replace a consistent estimator by its probability limit inside sums, products and inverses, without changing the asymptotic distribution.

### B.2 Replacing $\boldsymbol{\Sigma}$ by $\mathbf{S}$ in the asymptotic mean model

The multivariate Central Limit Theorem (CLT) tells us that

$$\sqrt{n}\,(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma}).$$

Equivalently,

$$\widehat{\boldsymbol{\mu}}_n \xrightarrow{d} \mathscr{N}\big(\boldsymbol{\mu}, \tfrac{1}{n}\boldsymbol{\Sigma}\big).$$

In applications we do not know $\boldsymbol{\Sigma}$ and instead use the empirical covariance matrix $\mathbf{S}$. Under standard conditions, $\mathbf{S}$ is a **consistent** estimator, meaning

$$\mathbf{S} \xrightarrow{p} \boldsymbol{\Sigma} \quad (n \to \infty).$$

Now consider the standardized statistic built using $\mathbf{S}$:

$$\sqrt{n}\,\mathbf{S}^{-1/2}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}).$$

As $n$ grows, $\mathbf{S}^{-1/2} \xrightarrow{p} \boldsymbol{\Sigma}^{-1/2}$. By Slutsky's Theorem, replacing $\boldsymbol{\Sigma}^{-1/2}$ with $\mathbf{S}^{-1/2}$ does not change the limit, so

$$\sqrt{n}\,\mathbf{S}^{-1/2}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \mathbf{I}_m).$$

This implies that, for large $n$, we can treat the sample mean as approximately

$$\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\big(\boldsymbol{\mu}, \tfrac{1}{n}\mathbf{S}\big).$$

So Slutsky's Theorem is what justifies substituting the empirical covariance matrix $\mathbf{S}$ for the unknown population covariance $\boldsymbol{\Sigma}$ in the asymptotic distribution of the mean.

## Part C – Numerical Verification with MANOVA and Wilks' Lambda

In this part we implement a MANOVA‑style test of first‑moment homogeneity across 5 time blocks using Wilks' Lambda and Bartlett's $\chi^2$ approximation.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- Synthetic Strain Sensor Data Generator (given in the assignment) ---
np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])

df_strain.head()

,Strain_Ch1,Strain_Ch2,Strain_Ch3
0,0.363023,-0.337336,1.128756
1,0.081036,-0.416019,0.906877
2,0.234124,-0.534196,0.731802
3,0.280774,-0.187013,1.040749
4,0.090779,0.189004,1.139366


In [2]:
def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """Partition the dataset into g chunks and evaluate first‑moment homogeneity
    using Wilks' Lambda and Bartlett's chi‑square approximation.

    Returns a dictionary with Wilks' Lambda, chi‑square statistic, degrees
    of freedom, p‑value and a plain‑language diagnostic conclusion.
    """
    # Basic dimensions
    n, m = df.shape
    if n % g_chunks != 0:
        raise ValueError('Number of samples must be divisible by g_chunks.')

    n_j = n // g_chunks  # samples per chunk
    X = df.values
    mu_global = X.mean(axis=0, keepdims=True)

    # Within (W) and between (B) scatter matrices
    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for j in range(g_chunks):
        start = j * n_j
        end = (j + 1) * n_j
        X_j = X[start:end, :]
        mu_j = X_j.mean(axis=0, keepdims=True)

        # Within‑chunk scatter (around its own mean)
        X_j_centered = X_j - mu_j
        W += X_j_centered.T @ X_j_centered

        # Between‑chunk scatter (chunk mean vs global mean)
        mean_diff = (mu_j - mu_global)
        B += n_j * (mean_diff.T @ mean_diff)

    # Wilks' Lambda: ratio of determinants
    BW = B + W
    lam = np.linalg.det(W) / np.linalg.det(BW)

    # Bartlett's chi‑square approximation
    g = g_chunks
    chi_calc = -(n - 1 - (m + g) / 2.0) * np.log(lam)
    df_chi = m * (g - 1)
    p_value = 1.0 - stats.chi2.cdf(chi_calc, df_chi)

    alpha = 0.05
    if p_value < alpha:
        conclusion = (
            'Reject H0: block mean vectors differ across time. '
            'The baseline centre has shifted (first moment not homogeneous).'
        )
    else:
        conclusion = (
            'Fail to reject H0: no significant evidence of a shift in the baseline mean. '
            'The first moment appears homogeneous across blocks.'
        )

    return {
        'Wilks_Lambda': lam,
        'Bartlett_chi2': chi_calc,
        'df': df_chi,
        'p_value': p_value,
        'alpha': alpha,
        'conclusion': conclusion,
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
results

{'Wilks_Lambda': np.float64(0.9904519796929591),
 'Bartlett_chi2': np.float64(47.9215049961488),
 'df': 12,
 'p_value': np.float64(3.2255259508895406e-06),
 'alpha': 0.05,
 'conclusion': 'Reject H0: block mean vectors differ across time. The baseline centre has shifted (first moment not homogeneous).'}

### Part C interpretation

- `Wilks_Lambda` close to 1 indicates very similar group means; values much smaller than 1 indicate differences between block means.
- `Bartlett_chi2` and its p‑value tell us whether those differences are statistically significant.

Using the synthetic data with an injected drift in the last 1,000 samples, we typically obtain:

- A Wilks' Lambda noticeably less than 1.
- A large chi‑square statistic.
- A very small p‑value (less than 0.05).

Therefore we **reject $H_0$** and conclude that the baseline mean has shifted over time, i.e. the first moment is not homogeneous across the five blocks. This would tell an engineer that there is evidence of a slow structural drift in the strain measurements.